<a href="https://colab.research.google.com/github/DimasBagusSusilo01/belajarAI/blob/main/belajarNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#===TEXT CLASSIFIER KALIMAT SARKAS===

In [ ]:
#sel ini mengimport alat yang dibutuhkan untuk membuat text classifier
import tensorflow as tf #Alat utama membangun NLP Text Classifier
from tensorflow import keras #Untuk memudahkan membangun model dari dataset
from tensorflow.keras.preprocessing.text import Tokenizer #Untuk membuat tiap kalimat pada dataset menjadi token untuk diolah menjadi model
from tensorflow.keras.preprocessing.sequence import pad_sequences #Untuk membuat dimensi antar kalimat menjadi sama

In [ ]:
import json#mengimport json untuk mengolah dataset json

kalimat = []
label = []

#Untuk membaca dataset dan memasukkan kalimat serta label ke variabel yang sesuai
with open("Sarcasm_Headlines_Dataset.json", 'r') as f:
  for line in f:
    item = json.loads(line)
    kalimat.append(item['headline'])
    label.append(item['is_sarcastic'])

print(f"Successfully loaded {len(kalimat)} items.")

Successfully loaded 28619 items.


In [ ]:
#Untuk mengubah kalimat menjadi token supaya mudah diolah
kosakata = 10000
oov_tok = "<OOV>"
tokenizer2 = Tokenizer(num_words=kosakata, oov_token=oov_tok)
ukuran_latihan = 20000

#Untuk membagi kalimat ujian dan kalimat latihan berdasarkan ukuran_latihan
kalimat_latihan = kalimat[0:ukuran_latihan]
kalimat_ujian = kalimat[ukuran_latihan:]
label_latihan = label[0:ukuran_latihan]
label_ujian = label[ukuran_latihan:]
print(len(kalimat_latihan))
print(len(kalimat_ujian))

#Untuk mengubah urutan kalimat latihan menjadi token dan mempelajarinya
panjangmaks = 100
trunc_type='post'
tokenizer2.fit_on_texts(kalimat_latihan)
indeks2 = tokenizer2.word_index
urutanlatihan = tokenizer2.texts_to_sequences(kalimat_latihan)
pad_latihan = pad_sequences(urutanlatihan, maxlen=panjangmaks, padding='post', truncating=trunc_type)

#Untuk mengubah urutan kalimat ujian menjadi token
urutanujian = tokenizer2.texts_to_sequences(kalimat_ujian)
pad_ujian = pad_sequences(urutanujian, maxlen=panjangmaks, padding='post', truncating=trunc_type)



20000
8619


In [ ]:
#Bagian ini untuk mengubah label latihan dan ujian menjadi array numpy supaya bisa diolah
import numpy as np
label_latihan = np.array(label_latihan)
label_ujian = np.array(label_ujian)
print(f"Type of label_latihan: {type(label_latihan)}")
print(f"Type of label_ujian: {type(label_ujian)}")

Type of label_latihan: <class 'numpy.ndarray'>
Type of label_ujian: <class 'numpy.ndarray'>


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(kosakata,16,input_length=panjangmaks),#untuk mengubah indeks kata hasil token menjadi dimensi tetap vektor padat untuk diolah
    tf.keras.layers.GlobalAveragePooling1D(),#mengambil rata rata tiap 100vektor 16dimensi menjadi satuvektor 16dimensi
    #2 baris dibawah melakukan proses sigmoid dan ReLu
    tf.keras.layers.Dense(24,activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam',metrics=['accuracy'])
kesimpulan = 30

#Ini adalah proses pelatihan model
total = model.fit(pad_latihan, label_latihan, epochs=kesimpulan,
                  validation_data=(pad_ujian, label_ujian), verbose=2)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


625/625 - 4s - 7ms/step - accuracy: 0.5620 - loss: 0.6809 - val_accuracy: 0.7764 - val_loss: 0.6278
Epoch 2/30
625/625 - 3s - 5ms/step - accuracy: 0.7476 - loss: 0.5260 - val_accuracy: 0.6604 - val_loss: 0.5798
Epoch 3/30
625/625 - 3s - 5ms/step - accuracy: 0.8163 - loss: 0.4102 - val_accuracy: 0.7761 - val_loss: 0.4426
Epoch 4/30
625/625 - 4s - 6ms/step - accuracy: 0.8461 - loss: 0.3551 - val_accuracy: 0.8455 - val_loss: 0.3576
Epoch 5/30
625/625 - 3s - 5ms/step - accuracy: 0.8655 - loss: 0.3177 - val_accuracy: 0.8196 - val_loss: 0.3825
Epoch 6/30
625/625 - 5s - 8ms/step - accuracy: 0.8775 - loss: 0.2952 - val_accuracy: 0.8160 - val_loss: 0.3934
Epoch 7/30
625/625 - 4s - 6ms/step - accuracy: 0.8873 - loss: 0.2738 - val_accuracy: 0.7935 - val_loss: 0.4475
Epoch 8/30
625/625 - 3s - 5ms/step - accuracy: 0.9005 - loss: 0.2459 - val_accuracy: 0.8089 - val_loss: 0.4233
Epoch 9/30
625/625 - 3s - 5ms/step - accuracy: 0.9074 - loss: 0.2300 - val_accuracy: 0.7858 - val_loss: 0.5104
Epoch 10/30


In [ ]:
#Ini adalah bagian uji coba model
sarkas_mulai = [
    'cool, i want to vomit'
]
urutan = tokenizer2.texts_to_sequences(sarkas_mulai)
pad3 = pad_sequences(urutan, maxlen=panjangmaks, padding='post', truncating=trunc_type)
print(model.predict(pad3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
[[0.8600136]]


In [ ]:
model.save('penggolong_sarkas.h5')

In [ ]:
import pickle
with open('tokenizer.pickle', 'wb') as f:
  pickle.dump(tokenizer2, f)